<a href="https://colab.research.google.com/github/MihalkaProMuzon/Real-Time-TTS-with-translate/blob/main/%D0%9F%D0%B5%D1%80%D0%B5%D0%B2%D0%BE%D0%B4_%D0%B2_%D1%80%D0%B5%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D0%BC_%D0%B2%D1%80%D0%B5%D0%BC%D0%B5%D0%BD%D0%B8_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Описание стенда

**Рабочий прототип системы потокового распознавания речи с последующим переводом и синтезом речи**, работающий в режиме реального времени через веб-интерфейс. Система реализует полный конвейер обработки аудиосигнала: от захвата звука микрофона в браузере до воспроизведения синтезированной переведённой речи.

Прототип реализован на основе **архитектуры потоковой обработки аудио**, где каждая стадия обработки выполняется независимо и взаимодействует через очереди сообщений.

Основная функциональность системы включает:

* потоковую передачу аудио с микрофона пользователя,
* онлайн-распознавание речи,
* автоматический перевод распознанного текста,
* синтез речи на основе переведённого текста,
* клонирование голоса пользователя,
* воспроизведение результата непосредственно в браузере.

Система реализована как **веб-приложение**, состоящее из клиентской части (браузер) и серверной части (Python backend).




<h1> Архитектура системы </h1>

Общая архитектура системы построена по принципу **конвейера обработки данных**. Аудиосигнал проходит несколько этапов обработки:

1. Захват аудио в браузере
2. Передача аудио на сервер
3. Потоковое распознавание речи
4. Формирование устойчивых текстовых фраз
5. Машинный перевод
6. Синтез речи
7. Передача аудио обратно клиенту
8. Последовательное воспроизведение

Архитектурно система реализована как набор независимых модулей:

* **Web-интерфейс**
* **Streaming ASR модуль**
* **Модуль перевода**
* **Модуль синтеза речи**
* **Система очередей**
* **Система отладки**

Между компонентами используются **потокобезопасные очереди Python (`queue.Queue`)**, что позволяет разделить вычисления между потоками и избежать блокировок.

<h1> Клиентская часть </h1>

Клиентская часть реализована на **JavaScript и Web Audio API** и выполняет следующие задачи:

* получение аудио с микрофона пользователя,
* потоковую передачу аудио на сервер,
* запись аудио в файл,
* получение отладочной информации,
* воспроизведение синтезированной речи.

<h3> Захват аудио </h3>

Аудиосигнал захватывается с помощью API:

```
navigator.mediaDevices.getUserMedia()
```

Полученный поток подключается к **AudioWorklet**, который выполняет разбивку аудио на пакеты фиксированного размера.

Использование AudioWorklet позволяет:

* получать аудио без задержек,
* работать с PCM-данными напрямую,
* избежать блокировки основного потока браузера.

Аудио передается на сервер в формате:

```
float32 PCM
mono
16000 Hz
```

Пакеты отправляются через WebSocket соединение.

<h1> Потоковая передача аудио </h1>

Для передачи данных используются **два WebSocket соединения**:

1. `/ws_audio` — поток PCM аудио для распознавания речи
2. `/ws_file` — запись аудио в файл для последующего прослушивания

Разделение потоков позволяет одновременно:

* выполнять потоковое распознавание,
* сохранять исходную запись.

<h1> Потоковое распознавание речи </h1>

Для распознавания речи используется модель:

**Faster-Whisper**

Это оптимизированная реализация модели Whisper, обеспечивающая более высокую скорость работы.

Модель загружается при запуске сервера:

```
WhisperModel("small")
```

Используется режим:

* `beam_size = 2`
* `temperature = 0`
* `word_timestamps = True`

Это позволяет получать **временные метки каждого слова**, что необходимо для потоковой обработки.

<h1> Оконная обработка аудио </h1>

Аудиосигнал обрабатывается методом **скользящего окна**.

Параметры обработки:

* длина окна — 2 секунды
* шаг окна — 0.35 секунды
* дополнительный padding — 0.25 секунды

Такой подход обеспечивает баланс между:

* стабильностью распознавания
* низкой задержкой

Каждое окно аудио передается в модель Whisper для распознавания.

<h1> Детекция речи </h1>

Перед запуском распознавания применяется **простая VAD (Voice Activity Detection)**.

Используются два признака:

1. энергия сигнала
2. коэффициент пересечения нуля (Zero Crossing Rate)

Это позволяет:

* игнорировать участки тишины
* уменьшить нагрузку на модель

<h1> Стабилизация гипотез распознавания </h1>

При потоковом распознавании Whisper может изменять предыдущие слова при обработке новых фрагментов аудио.

Для решения этой проблемы используется механизм **буфера гипотез**.

Хранятся последние несколько гипотез распознавания:

```
HYP_HISTORY = 4
```

Далее применяется алгоритм **голосования по нескольким гипотезам**.

Алгоритм:

1. выбирается последняя гипотеза
2. она выравнивается с предыдущими гипотезами
3. определяется общий стабильный префикс

Только слова, подтвержденные несколькими гипотезами, считаются окончательными.

Это позволяет значительно уменьшить:

* повторения слов
* исправления текста
* нестабильность вывода.

<h1> Формирование фраз </h1>

После подтверждения отдельных слов система объединяет их в **логические фразы**.

Фраза завершается если выполняется одно из условий:

* пауза между словами больше заданного порога
* достигнута максимальная длина фразы
* превышена максимальная длительность фразы

Этот механизм позволяет формировать удобные для перевода текстовые блоки.


<h1> Машинный перевод </h1>

Для перевода текста используется модель:

```
Helsinki-NLP OPUS-MT
```

Поддерживаются направления:

* русский → английский
* английский → русский

Модели загружаются через библиотеку **Transformers**.

Перевод выполняется отдельным рабочим потоком, что позволяет выполнять его параллельно с распознаванием речи.

<h1> Синтез речи </h1>

Для генерации аудио используется модель:

```
XTTS v2
```

из библиотеки **Coqui TTS**.

Эта модель поддерживает:

* мультиязычный синтез речи
* клонирование голоса по аудио-референсу

Синтез выполняется на GPU при наличии CUDA.

<h1> Клонирование голоса </h1>

Система поддерживает **калибровку голоса пользователя**.

Пользователь записывает короткий образец речи (~15 секунд), который сохраняется как референсный файл.

Этот файл используется XTTS в параметре:

```
speaker_wav
```

Это позволяет синтезировать переведенную речь **с тембром, похожим на голос пользователя**.

<h1> Очереди обработки </h1>

Обработка данных разделена на несколько этапов с использованием очередей:

```
audio_q   → ASR
phrase_q  → translation
tts_q     → TTS
```

Каждый этап выполняется в отдельном потоке.

Это позволяет:

* избежать блокировок
* равномерно распределять нагрузку
* масштабировать систему.

<h1> Система воспроизведения </h1>

Синтезированная речь передается клиенту в формате:

```
base64 encoded WAV
```

В браузере используется очередь воспроизведения аудио.

Это предотвращает наложение нескольких аудиофрагментов и обеспечивает последовательное воспроизведение.


<h1> Система отладки </h1>

Для мониторинга работы системы реализован отдельный канал WebSocket `/debug`.

Через него передаются события:

* распознавание слов
* формирование фраз
* перевод
* синтез речи
* ошибки системы

Это позволяет наблюдать за работой конвейера обработки в реальном времени.

<h1> Развертывание системы </h1>

Серверная часть реализована на базе:

```
FastAPI
Uvicorn
```

Для публикации демо используется SSH-туннель через сервис:

```
localhost.run
```

<h1> Назначение прототипа </h1>

Разработанная система является **рабочим прототипом**, демонстрирующим возможность реализации потокового переводчика речи с использованием современных моделей машинного обучения.

Прототип реализует полный цикл обработки речи:

* потоковый захват аудио
* онлайн-распознавание
* перевод текста
* синтез речи
* клонирование голоса

В дальнейшем система может быть улучшена за счёт:

* более точных методов VAD
* улучшенных моделей перевода
* оптимизации задержек
* поддержки дополнительных языков.

# Код приложения

In [ ]:
if 'installed' not in globals():
    installed = False

if not installed:
    !pip install -q fastapi uvicorn pyngrok nest_asyncio faster-whisper soxr coqui-tts transformers sentencepiece
    installed = True

Импорты

In [ ]:
import os
import json
import time
import queue
import asyncio
import threading
from collections import Counter, deque

import nest_asyncio
nest_asyncio.apply()

import numpy as np
import soxr

from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse, FileResponse, JSONResponse
from faster_whisper import WhisperModel
from fastapi import Request

import uvicorn

import torch
from TTS.api import TTS

import soundfile as sf
import io
import base64

from transformers import MarianMTModel, MarianTokenizer

print("CUDA:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CUDA: True


Конфигурация

In [ ]:
# =========================================================
# AUDIO CONFIG
# =========================================================

TARGET_SR = 16000

WINDOW_SEC = 2.0
STEP_SEC = 0.35
PAD_SEC = 0.25
RIGHT_GUARD_SEC = 0.25

WINDOW_SAMPLES = int(WINDOW_SEC * TARGET_SR)
STEP_SAMPLES = int(STEP_SEC * TARGET_SR)
PAD_SAMPLES = int(PAD_SEC * TARGET_SR)


# =========================================================
# PHRASE BUILDER
# =========================================================

PAUSE_SEC = 1.2
MAX_PHRASE_WORDS = 12
MAX_PHRASE_SEC = 3.6


# =========================================================
# SIMPLE VAD
# =========================================================

VAD_ENERGY_THRESHOLD = 0.003
VAD_ZCR_THRESHOLD = 0.20


# =========================================================
# HYPOTHESIS BUFFER
# =========================================================

# hypothesis ring buffer
HYP_HISTORY = 4              # сколько последних окон хранить
MIN_STABLE_MATCHES = 2       # минимум совпадений для слова
MAX_ALIGN_OFFSET = 3         # разрешённый сдвиг при выравнивании
MAX_TAIL_KEEP = 1            # не коммитим последний 1 токен agreed prefix


# =========================================================
# FILES
# =========================================================

REC_FILE = "recorded.webm"

# =========================================================
# WHISPER
# =========================================================

MODEL_NAME = "small"

COMPUTE_TYPE = "float16"

# =========================================================
# TTS
# =========================================================
TTS_MODEL = TTS(
    "tts_models/multilingual/multi-dataset/xtts_v2"
).to(DEVICE)

SPEAKER_REF = "reference.wav"


# =========================================================
# TRANSLATE
# =========================================================

TRANSLATE_DIRECTION = "ru-en"

# RU -> EN
MODEL_RU_EN = "Helsinki-NLP/opus-mt-ru-en"

tokenizer_ru_en = MarianTokenizer.from_pretrained(MODEL_RU_EN)
model_ru_en = MarianMTModel.from_pretrained(MODEL_RU_EN).to(DEVICE)

# EN -> RU
MODEL_EN_RU = "Helsinki-NLP/opus-mt-en-ru"

tokenizer_en_ru = MarianTokenizer.from_pretrained(MODEL_EN_RU)
model_en_ru = MarianMTModel.from_pretrained(MODEL_EN_RU).to(DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Окружение

In [ ]:
# =========================================================
# RUNTIME
# =========================================================
app = FastAPI()

audio_q = queue.Queue(maxsize=512)

phrase_q = queue.Queue(maxsize=128)
tts_q = queue.Queue(maxsize=64)


debug_clients = set()
debug_lock = asyncio.Lock()

MAIN_LOOP = None
ASR_THREAD = None
ASR_STOP = threading.Event()
WHISPER_MODEL = None
ASR_LOCK = threading.Lock()


# =========================================================
# DEBUG
# =========================================================
async def _broadcast_debug(payload: dict):
    dead = []
    async with debug_lock:
        for ws in debug_clients:
            try:
                await ws.send_text(json.dumps(payload, ensure_ascii=False))
            except Exception:
                dead.append(ws)
        for ws in dead:
            debug_clients.discard(ws)


def push_debug(kind: str, text: str, **extra):
    payload = {
        "type": kind,
        "text": text,
        "ts": time.time(),
    }
    payload.update(extra)

    if MAIN_LOOP is not None:
        asyncio.run_coroutine_threadsafe(_broadcast_debug(payload), MAIN_LOOP)


# =========================================================
# HELPERS
# =========================================================
def bytes_to_float32_pcm(raw: bytes) -> np.ndarray:
    return np.frombuffer(raw, dtype=np.float32).astype(np.float32, copy=False)


def resample_audio(audio: np.ndarray, src_sr: int, dst_sr: int = TARGET_SR) -> np.ndarray:
    audio = audio.astype(np.float32, copy=False)
    if src_sr == dst_sr:
        return audio
    return soxr.resample(audio, src_sr, dst_sr, quality="HQ").astype(np.float32, copy=False)


def smart_join(tokens):
    out = ""
    punct = {".", ",", "!", "?", ":", ";", "…"}
    for tok in tokens:
        tok = tok.strip()
        if not tok:
            continue

        if not out:
            out = tok
        elif tok in punct:
            out += tok
        else:
            out += " " + tok
    return out.strip()


def normalize_token(t: str) -> str:
    return t.strip().lower()


def token_is_punct(t: str) -> bool:
    return normalize_token(t) in {".", ",", "!", "?", ":", ";", "…"}


def zero_crossing_rate(x: np.ndarray) -> float:
    if x.size < 2:
        return 0.0
    return float(np.mean(np.abs(np.diff(np.signbit(x).astype(np.int8)))))


def is_speech_like(frame: np.ndarray) -> bool:
    energy = float(np.mean(np.abs(frame)))
    zcr = zero_crossing_rate(frame)
    return energy >= VAD_ENERGY_THRESHOLD and zcr <= VAD_ZCR_THRESHOLD


def clear_queue_safely(q: queue.Queue):
    try:
        while True:
            q.get_nowait()
    except queue.Empty:
        pass


def best_aligned_prefix(base_words, other_words, max_offset=3):
    """
    Ищет лучший общий префикс между двумя гипотезами с небольшим сдвигом.
    Возвращает список слов из other_words.
    """
    best = []

    max_base_off = min(max_offset, len(base_words))
    max_other_off = min(max_offset, len(other_words))

    for boff in range(max_base_off + 1):
        for ooff in range(max_other_off + 1):
            tmp = []
            i = 0
            while boff + i < len(base_words) and ooff + i < len(other_words):
                a = normalize_token(base_words[boff + i]["text"])
                b = normalize_token(other_words[ooff + i]["text"])
                if a != b:
                    break
                tmp.append(other_words[ooff + i])
                i += 1

            if len(tmp) > len(best):
                best = tmp

    return best


def build_voted_agreement(history, max_offset=3):
    """
    history: список гипотез (каждая — список слов)
    Ищем устойчивый префикс через несколько последних окон.
    Берём последнюю гипотезу как опорную, а затем считаем,
    сколько раз каждый токен на позиции подтверждён соседними окнами.
    """
    if len(history) < 2:
        return []

    base = history[-1]
    if not base:
        return []

    aligned_prefixes = []
    for hyp in history[:-1]:
        pref = best_aligned_prefix(base, hyp, max_offset=max_offset)
        if pref:
            aligned_prefixes.append(pref)

    if not aligned_prefixes:
        return []

    voted = []
    for pos in range(len(base)):
        tok = normalize_token(base[pos]["text"])
        votes = 1  # базовая гипотеза сама за себя

        for pref in aligned_prefixes:
            if pos < len(pref) and normalize_token(pref[pos]["text"]) == tok:
                votes += 1

        if votes >= MIN_STABLE_MATCHES:
            voted.append(base[pos])
        else:
            break

    return voted

def get_asr_language():
    if TRANSLATE_DIRECTION == "ru-en":
        return "ru"
    if TRANSLATE_DIRECTION == "en-ru":
        return "en"
    return "ru"

def get_tts_language():
    if TRANSLATE_DIRECTION == "ru-en":
        return "en"
    if TRANSLATE_DIRECTION == "en-ru":
        return "ru"

    return "ru"

def translate_text(text: str, direction: str):
    try:
        if direction == "ru-en":
            tokenizer = tokenizer_ru_en
            model = model_ru_en
        elif direction == "en-ru":
            tokenizer = tokenizer_en_ru
            model = model_en_ru
        else:
            push_debug("error", f"Unknown translation direction: {direction}")
            return text

        batch = tokenizer(
            [text],
            return_tensors="pt",
            padding=True
        ).to(DEVICE)

        generated = model.generate(**batch)

        translated = tokenizer.decode(
            generated[0],
            skip_special_tokens=True
        )

        if not translated.strip():
            translated = text

        push_debug("info", f"[TRANSLATE] {translated}")

        return translated

    except Exception as e:
        push_debug("error", f"Translation error: {e}")
        return text




Translation worker

In [ ]:
def translation_worker():
    push_debug("info", "Translation worker started")

    while True:
        try:
            text, direction = phrase_q.get(timeout=1)
        except queue.Empty:
            continue

        try:
            translated = translate_text(text, direction)

            if not translated:
                translated = text

            try:
                tts_q.put_nowait((translated, direction))
            except queue.Full:
                push_debug("error", "tts_q full")

        except Exception as e:
            push_debug("error", f"Translation worker error: {e}")

        finally:
            phrase_q.task_done()

tts worker

In [ ]:
def tts_worker():
    push_debug("info", "TTS worker started")

    while True:
        try:
            text, direction = tts_q.get(timeout=1)
        except queue.Empty:
            continue

        try:
            synthesize_tts(text, direction)

        except Exception as e:
            push_debug("error", f"TTS worker error: {e}")

        finally:
            tts_q.task_done()

Speech Pipeline

In [ ]:
def synthesize_tts(text, direction):
    try:
        if not text.strip():
            return

        if not os.path.exists(SPEAKER_REF):
            push_debug("error","Reference voice missing")
            return

        language = "en" if direction == "ru-en" else "ru"

        audio = TTS_MODEL.tts(
            text=text,
            speaker_wav=SPEAKER_REF,
            language=language,
            speed=1.15
        )

        wav_bytes = io.BytesIO()
        sf.write(wav_bytes, audio, 24000, format="WAV")

        wav_bytes.seek(0)

        encoded = base64.b64encode(wav_bytes.read()).decode("utf-8")

        payload = {
            "type": "tts_audio",
            "audio_base64": encoded,
            "text": text
        }

        if MAIN_LOOP:
            asyncio.run_coroutine_threadsafe(
                _broadcast_debug(payload),
                MAIN_LOOP
            )

    except Exception as e:
        push_debug("error", f"TTS error: {e}")

Сервер Фастапи

In [ ]:
# =========================================================
# STREAMING ASR
# =========================================================
class StreamingASR:
    def __init__(self, model: WhisperModel):
        self.model = model
        self.audio_buffer = np.zeros((0,), dtype=np.float32)
        self.sample_cursor = 0
        self.last_committed_end = -1e9

        # ring buffer гипотез
        self.hyp_history = deque(maxlen=HYP_HISTORY)

        # phrase builder
        self.phrase_tokens = []
        self.phrase_start_time = None
        self.last_word_end = None

        # чтобы не заспамить дебаг
        self.last_vad_state = None

    def flush_phrase(self):
        if not self.phrase_tokens:
            return

        text = smart_join(self.phrase_tokens)
        if text.strip():
            try:
                phrase_q.put_nowait((text, TRANSLATE_DIRECTION))
            except queue.Full:
                push_debug("error", "phrase_q full")

        self.phrase_tokens = []
        self.phrase_start_time = None
        self.last_word_end = None

    def absorb_word(self, w):
        if self.phrase_start_time is None:
            self.phrase_start_time = w["start"]

        if self.last_word_end is not None and (w["start"] - self.last_word_end) > PAUSE_SEC:
            self.flush_phrase()
            self.phrase_start_time = w["start"]

        token = w["text"]

        if self.phrase_tokens:
            last = self.phrase_tokens[-1].strip().lower()

            if normalize_token(token) == normalize_token(last):
                return

        self.phrase_tokens.append(w["text"])
        self.last_word_end = w["end"]

        if len(self.phrase_tokens) >= MAX_PHRASE_WORDS:
            self.flush_phrase()
            return

        if self.phrase_start_time is not None:
            if (w["end"] - self.phrase_start_time) >= MAX_PHRASE_SEC:
                self.flush_phrase()

    def accept_audio(self, pcm: np.ndarray):
        if pcm.size == 0:
            return

        if self.audio_buffer.size == 0:
            self.audio_buffer = pcm
        else:
            self.audio_buffer = np.concatenate([self.audio_buffer, pcm])

        while self.audio_buffer.size >= WINDOW_SAMPLES:
            self.process_window()
            self.audio_buffer = self.audio_buffer[STEP_SAMPLES:]
            self.sample_cursor += STEP_SAMPLES

            if self.last_word_end is not None:
                current_time = self.sample_cursor / TARGET_SR
                if (current_time - self.last_word_end) > PAUSE_SEC:
                    self.flush_phrase()

    def process_window(self):
        frame_start_time = self.sample_cursor / TARGET_SR
        frame = self.audio_buffer[:WINDOW_SAMPLES]

        if frame.size < WINDOW_SAMPLES:
            return

        push_debug(
            "info",
            f"window_start={frame_start_time:.2f}s, audio_buffer={self.audio_buffer.size / TARGET_SR:.2f}s"
        )

        # ---- VAD ----
        speech_like = is_speech_like(frame)
        if self.last_vad_state != speech_like:
            push_debug("info", f"vad={'speech' if speech_like else 'silence'}")
            self.last_vad_state = speech_like

        if not speech_like:
            return

        padded = np.pad(frame, (PAD_SAMPLES, PAD_SAMPLES)).astype(np.float32, copy=False)

        try:
            segments, _ = self.model.transcribe(
                padded,
                language=get_asr_language(),
                beam_size=2,
                best_of=2,
                temperature=0.0,
                word_timestamps=True,
                vad_filter=False,
                condition_on_previous_text=False,
            )
        except Exception as e:
            push_debug("error", f"Whisper transcribe error: {e}")
            return

        pad_sec = PAD_SAMPLES / TARGET_SR
        words = []

        for seg in segments:
            if not seg.words:
                continue

            for w in seg.words:
                token = (w.word or "").strip()
                if not token:
                    continue

                abs_start = frame_start_time + float(w.start) - pad_sec
                abs_end = frame_start_time + float(w.end) - pad_sec

                if abs_end <= abs_start:
                    continue

                words.append({
                    "text": token,
                    "start": abs_start,
                    "end": abs_end,
                })

        if not words:
            return

        commit_deadline = frame_start_time + WINDOW_SEC - RIGHT_GUARD_SEC

        current_hyp = [
            w for w in words
            if w["end"] > self.last_committed_end + 0.03
        ]

        if not current_hyp:
            return

        push_debug(
            "info",
            "HYP: " + " ".join(w["text"] for w in current_hyp[:14])
        )

        self.hyp_history.append(current_hyp)

        if len(self.hyp_history) < 2:
            return

        agreed = build_voted_agreement(list(self.hyp_history), max_offset=MAX_ALIGN_OFFSET)

        push_debug(
            "info",
            "AGR: " + " ".join(w["text"] for w in agreed[:14])
        )

        if len(agreed) > MAX_TAIL_KEEP:
            commit_candidates = agreed[:-MAX_TAIL_KEEP]
        else:
            commit_candidates = []

        # если ничего не коммитится — проверяем последнее слово
        if not commit_candidates and agreed:
            last = agreed[-1]

            if last["end"] <= commit_deadline - 0.15:
                commit_candidates = [last]

        # fallback для последнего слова, если оно давно стабильно и одно
        if not commit_candidates and len(agreed) == 1:
            w = agreed[0]
            if w["end"] <= commit_deadline - 0.15:
                commit_candidates = [w]

        push_debug(
            "info",
            "CMT: " + " ".join(w["text"] for w in commit_candidates[:14])
        )

        emitted = False

        for w in commit_candidates:
            token = normalize_token(w["text"])

            if self.phrase_tokens and token == normalize_token(self.phrase_tokens[-1]):
                continue

            if w["end"] <= commit_deadline and w["end"] > self.last_committed_end + 0.03:
                self.last_committed_end = w["end"]
                emitted = True

                push_debug(
                    "commit",
                    w["text"],
                    start=round(w["start"], 2),
                    end=round(w["end"], 2),
                )

                self.absorb_word(w)

        if not emitted and self.last_word_end is not None:
            if (commit_deadline - self.last_word_end) > PAUSE_SEC:
                self.flush_phrase()


def asr_worker():
    push_debug("info", "ASR worker started")
    asr = StreamingASR(WHISPER_MODEL)

    while not ASR_STOP.is_set():
        try:
            item = audio_q.get(timeout=0.2)
        except queue.Empty:
            continue

        if item is None:
            break

        try:
            asr.accept_audio(item)
        except Exception as e:
            push_debug("error", f"ASR worker error: {e}")

    asr.flush_phrase()
    push_debug("info", "ASR worker stopped")


def restart_asr_session():
    global ASR_THREAD

    with ASR_LOCK:
        ASR_STOP.set()
        try:
            audio_q.put_nowait(None)
        except Exception:
            pass

        if ASR_THREAD is not None and ASR_THREAD.is_alive():
            ASR_THREAD.join(timeout=1.0)

        clear_queue_safely(audio_q)

        ASR_STOP.clear()
        ASR_THREAD = threading.Thread(target=asr_worker, daemon=True)
        ASR_THREAD.start()

        # threading.Thread(target=translation_worker, daemon=True).start()
        # threading.Thread(target=tts_worker, daemon=True).start()


# =========================================================
# HTML
# =========================================================
HTML = r"""
<!DOCTYPE html>
<html lang="ru">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>Production-like Streaming Whisper Demo</title>
<style>
    body {
        font-family: Arial, sans-serif;
        max-width: 1050px;
        margin: 20px auto;
        padding: 0 16px;
        background: #111;
        color: #eee;
    }
    .card {
        background: #1b1b1b;
        border: 1px solid #333;
        border-radius: 12px;
        padding: 16px;
        margin-bottom: 16px;
    }
    .row { margin-bottom: 10px; }
    .muted { color: #aaa; font-size: 14px; }

    button {
        padding: 10px 16px;
        margin-right: 8px;
        border: 0;
        border-radius: 8px;
        cursor: pointer;
        font-size: 14px;
    }

    #start { background: #2d7ef7; color: white; }
    #stop { background: #c94b4b; color: white; }

    audio {
        width: 100%;
        margin-top: 10px;
    }

    #debugBox {
        height: 360px;
        overflow-y: auto;
        background: #0c0c0c;
        border: 1px solid #333;
        border-radius: 10px;
        padding: 10px;
        font-family: monospace;
        white-space: pre-wrap;
    }

    .dbg-line {
        padding: 4px 0;
        border-bottom: 1px solid #1f1f1f;
    }

    .dbg-commit { color: #79d279; }
    .dbg-phrase { color: #ffcc66; }
    .dbg-info { color: #8ab4f8; }
    .dbg-error { color: #ff6b6b; }
</style>
</head>
<body>
    <h2>Production-like Streaming Whisper Demo</h2>

    <div class="card">
        <div class="row"><b>Текущий микрофон:</b> <span id="micName">определяется...</span></div>
        <div class="row muted" id="micMeta"></div>

        <select id="direction">
        <option value="ru-en">RU → EN</option>
        <option value="en-ru">EN → RU</option>
        </select>

        <button id="start">Start Recording</button>
        <button id="stop">Stop</button>



        <audio id="player" controls></audio>
    </div>

    <div class="card">
    <h3>Voice calibration</h3>

    <div class="row">
    Запишите голос (~15 секунд) для референса синтеза.
    </div>

    <button id="calibStart">Start calibration</button>
    <button id="calibStop">Stop calibration</button>

    <audio id="calibPlayer" controls></audio>
    </div>

    <div class="card">
        <h3>Debug</h3>
        <div id="debugBox"></div>
    </div>

<script>
let calibRecorder = null;
let calibStream = null;
let calibChunks = [];

let ttsQueue = [];
let ttsPlaying = false;

let wsAudio = null;
let wsDebug = null;
let wsFile = null;

let mediaRecorder = null;
let mediaStream = null;

let audioContext = null;
let sourceNode = null;
let workletNode = null;

function wsProto() {
    return location.protocol === "https:" ? "wss://" : "ws://";
}

function playNextTTS() {

    if (ttsPlaying) return;
    if (ttsQueue.length === 0) return;

    ttsPlaying = true;

    const url = ttsQueue.shift();
    const audio = new Audio(url);

    audio.onended = () => {
        URL.revokeObjectURL(url);
        ttsPlaying = false;
        playNextTTS();
    };

    audio.onerror = () => {
        URL.revokeObjectURL(url);
        ttsPlaying = false;
        playNextTTS();
    };

    audio.play();
}

async function startCalibration() {
    try {
        calibStream = await navigator.mediaDevices.getUserMedia({audio:true})
        calibChunks = []
        calibRecorder = new MediaRecorder(calibStream)
        calibRecorder.ondataavailable = e=>{
            if(e.data.size>0) calibChunks.push(e.data)
        }
        calibRecorder.start()
        appendDebug("[CALIB] recording started","dbg-info")

    } catch(e){
        appendDebug("[CALIB ERROR] "+e,"dbg-error")
    }
}

async function stopCalibration(){
    if(!calibRecorder) return
    calibRecorder.stop()
    calibRecorder.onstop = async ()=>{

        const blob = new Blob(calibChunks,{type:"audio/webm"})
        const arrayBuffer = await blob.arrayBuffer()
        await fetch("/upload_reference",{
            method:"POST",
            body:arrayBuffer
        })

        const url = URL.createObjectURL(blob)
        const player = document.getElementById("calibPlayer")
        player.src = url
        appendDebug("[CALIB] reference saved","dbg-info")
        if(calibStream){
            calibStream.getTracks().forEach(t=>t.stop())
        }
    }
}

function appendDebug(text, cls="dbg-info") {
    const box = document.getElementById("debugBox");
    const div = document.createElement("div");
    div.className = "dbg-line " + cls;
    div.textContent = text;
    box.appendChild(div);
    box.scrollTop = box.scrollHeight;

    while (box.children.length > 350) {
        box.removeChild(box.firstChild);
    }
}

async function setupMicInfo() {
    try {
        const tmpStream = await navigator.mediaDevices.getUserMedia({ audio: true });
        const track = tmpStream.getAudioTracks()[0];
        const settings = track.getSettings ? track.getSettings() : {};
        const label = track.label || "Неизвестный микрофон";

        document.getElementById("micName").textContent = label;
        document.getElementById("micMeta").textContent =
            `sampleRate: ${settings.sampleRate || "?"}, channelCount: ${settings.channelCount || "?"}`;

        tmpStream.getTracks().forEach(t => t.stop());
    } catch (e) {
        document.getElementById("micName").textContent = "Не удалось получить доступ";
        document.getElementById("micMeta").textContent = String(e);
    }
}

async function connectDebug() {
    wsDebug = new WebSocket(wsProto() + location.host + "/debug");
    wsDebug.onopen = () => appendDebug("[INFO] Debug socket connected", "dbg-info");
    wsDebug.onclose = () => appendDebug("[INFO] Debug socket closed", "dbg-info");

    wsDebug.onmessage = (event) => {
        const msg = JSON.parse(event.data);

        if (msg.type === "commit") {
            appendDebug(`[COMMIT ${msg.start}–${msg.end}] ${msg.text}`, "dbg-commit");
        } else if (msg.type === "phrase") {
            appendDebug(`[PHRASE] ${msg.text}`, "dbg-phrase");
        } else if (msg.type === "error") {
            appendDebug(`[ERROR] ${msg.text}`, "dbg-error");
        } else if (msg.type === "tts_audio") {
            const audioData = atob(msg.audio_base64);
            const array = new Uint8Array(audioData.length);

            for (let i = 0; i < audioData.length; i++) {
                array[i] = audioData.charCodeAt(i);
            }

            const blob = new Blob([array], { type: "audio/wav" });
            const url = URL.createObjectURL(blob);

            const audio = new Audio(url);

            audio.onended = () => URL.revokeObjectURL(url);

            ttsQueue.push(url);
            playNextTTS();

            appendDebug(`[TTS] ${msg.text}`, "dbg-info");

        } else {
            appendDebug(`[INFO] ${msg.text}`, "dbg-info");
        }
    };
}

async function startRecording() {
    try {
        wsAudio = new WebSocket(wsProto() + location.host + "/ws_audio");
        wsAudio.binaryType = "arraybuffer";

        wsFile = new WebSocket(wsProto() + location.host + "/ws_file");
        wsFile.binaryType = "arraybuffer";

        await new Promise((resolve, reject) => {
            wsAudio.onopen = () => resolve();
            wsAudio.onerror = reject;
        });

        await new Promise((resolve, reject) => {
            wsFile.onopen = () => resolve();
            wsFile.onerror = reject;
        });

        const direction = document.getElementById("direction").value;

        wsAudio.send(JSON.stringify({
            type: "direction",
            value: direction
        }));

        mediaStream = await navigator.mediaDevices.getUserMedia({
            audio: {
                channelCount: 1,
                echoCancellation: false,
                noiseSuppression: false,
                autoGainControl: false
            }
        });

        // ---- file recording path ----
        let mimeType = "audio/webm;codecs=opus";
        if (!MediaRecorder.isTypeSupported(mimeType)) {
            mimeType = "audio/webm";
        }

        mediaRecorder = new MediaRecorder(mediaStream, {
            mimeType: mimeType,
            audioBitsPerSecond: 128000
        });

        mediaRecorder.ondataavailable = async (event) => {
            if (!event.data || event.data.size === 0) return;
            if (!wsFile || wsFile.readyState !== WebSocket.OPEN) return;

            const buf = await event.data.arrayBuffer();
            wsFile.send(buf);
        };

        mediaRecorder.start(500);

        // ---- streaming PCM path ----
        audioContext = new AudioContext({ sampleRate: 16000 });
        appendDebug("[INFO] audioContext.sampleRate=" + audioContext.sampleRate, "dbg-info");

        const PACKET_SAMPLES = 2048;

        const workletCode = `
            class PCMWorklet extends AudioWorkletProcessor {
                constructor() {
                    super();
                    this.packetSize = ${PACKET_SAMPLES};
                    this.buffer = new Float32Array(this.packetSize);
                    this.offset = 0;
                }

                process(inputs) {
                    const input = inputs[0];
                    if (!input || !input[0]) return true;

                    const channel = input[0];
                    let i = 0;

                    while (i < channel.length) {
                        const n = Math.min(channel.length - i, this.packetSize - this.offset);
                        this.buffer.set(channel.subarray(i, i + n), this.offset);
                        this.offset += n;
                        i += n;

                        if (this.offset === this.packetSize) {
                            const out = new Float32Array(this.buffer);
                            this.port.postMessage(out.buffer, [out.buffer]);
                            this.buffer = new Float32Array(this.packetSize);
                            this.offset = 0;
                        }
                    }

                    return true;
                }
            }

            registerProcessor('pcm-worklet', PCMWorklet);
        `;

        const blob = new Blob([workletCode], { type: "application/javascript" });
        const blobUrl = URL.createObjectURL(blob);
        await audioContext.audioWorklet.addModule(blobUrl);
        URL.revokeObjectURL(blobUrl);

        if (wsAudio && wsAudio.readyState === WebSocket.OPEN) {
            wsAudio.send(JSON.stringify({
                type: "config",
                sample_rate: audioContext.sampleRate,
                packet_samples: PACKET_SAMPLES
            }));
        }

        sourceNode = audioContext.createMediaStreamSource(mediaStream);
        workletNode = new AudioWorkletNode(audioContext, "pcm-worklet");

        workletNode.port.onmessage = (event) => {
            if (!wsAudio || wsAudio.readyState !== WebSocket.OPEN) return;
            wsAudio.send(event.data);
        };

        sourceNode.connect(workletNode);

        const muteGain = audioContext.createGain();
        muteGain.gain.value = 0.0;
        workletNode.connect(muteGain);
        muteGain.connect(audioContext.destination);

        appendDebug("[INFO] Recording started", "dbg-info");
    } catch (e) {
        appendDebug("[ERROR] Start failed: " + e, "dbg-error");
    }
}

async function stopRecording() {
    try {
        if (mediaRecorder && mediaRecorder.state !== "inactive") {
            mediaRecorder.stop();
        }

        if (workletNode) {
            workletNode.disconnect();
            workletNode = null;
        }

        if (sourceNode) {
            sourceNode.disconnect();
            sourceNode = null;
        }

        if (audioContext) {
            await audioContext.close();
            audioContext = null;
        }

        if (mediaStream) {
            mediaStream.getTracks().forEach(t => t.stop());
            mediaStream = null;
        }

        setTimeout(() => {
            if (wsAudio && wsAudio.readyState === WebSocket.OPEN) wsAudio.close();
            if (wsFile && wsFile.readyState === WebSocket.OPEN) wsFile.close();
        }, 400);

        setTimeout(() => {
            const player = document.getElementById("player");
            player.src = "/audio?ts=" + Date.now();
            player.load();
        }, 1200);

        appendDebug("[INFO] Recording stopped", "dbg-info");
    } catch (e) {
        appendDebug("[ERROR] Stop failed: " + e, "dbg-error");
    }
}

document.getElementById("start").onclick = startRecording;
document.getElementById("stop").onclick = stopRecording;


document.getElementById("calibStart").onclick = startCalibration
document.getElementById("calibStop").onclick = stopCalibration

setupMicInfo();
connectDebug();
</script>
</body>
</html>
"""


# =========================================================
# STARTUP / SHUTDOWN
# =========================================================
@app.on_event("startup")
async def on_startup():
    global MAIN_LOOP, WHISPER_MODEL, ASR_THREAD
    MAIN_LOOP = asyncio.get_running_loop()

    push_debug("info", "Loading Whisper model...")
    WHISPER_MODEL = WhisperModel(
        MODEL_NAME,
        device=DEVICE,
        compute_type=COMPUTE_TYPE,
    )
    push_debug("info", f"Whisper ready: {MODEL_NAME}")

    ASR_STOP.clear()
    ASR_THREAD = threading.Thread(target=asr_worker, daemon=True)
    ASR_THREAD.start()

    threading.Thread(target=translation_worker, daemon=True).start()
    threading.Thread(target=tts_worker, daemon=True).start()


@app.on_event("shutdown")
async def on_shutdown():
    ASR_STOP.set()
    try:
        audio_q.put_nowait(None)
    except Exception:
        pass


# =========================================================
# ROUTES
# =========================================================
@app.get("/")
async def index():
    return HTMLResponse(HTML)


@app.get("/health")
async def health():
    return JSONResponse({"ok": True})


@app.get("/audio")
async def get_audio():
    if not os.path.exists(REC_FILE):
        return HTMLResponse("No audio yet", status_code=404)
    return FileResponse(REC_FILE, media_type="audio/webm", filename="recorded.webm")


@app.websocket("/debug")
async def debug_ws(ws: WebSocket):
    await ws.accept()
    async with debug_lock:
        debug_clients.add(ws)

    try:
        await ws.send_text(json.dumps({"type": "info", "text": "Debug channel connected"}, ensure_ascii=False))
        while True:
            await asyncio.sleep(60)
    except WebSocketDisconnect:
        pass
    finally:
        async with debug_lock:
            debug_clients.discard(ws)


@app.websocket("/ws_file")
async def ws_file(ws: WebSocket):
    await ws.accept()
    push_debug("info", "File socket connected")

    f = open(REC_FILE, "wb")
    try:
        while True:
            data = await ws.receive_bytes()
            f.write(data)
            f.flush()
    except WebSocketDisconnect:
        pass
    except Exception as e:
        push_debug("error", f"File socket error: {e}")
    finally:
        f.close()
        push_debug("info", "Audio file closed")


@app.websocket("/ws_audio")
async def ws_audio(ws: WebSocket):
    global TRANSLATE_DIRECTION

    await ws.accept()
    push_debug("info", "Audio socket connected")

    restart_asr_session()

    src_sr = TARGET_SR
    packet_samples = None
    received_samples = 0


    try:
        while True:
            try:
                msg = await ws.receive()
            except WebSocketDisconnect:
                break

            if msg.get("type") == "websocket.disconnect":
                break

            # ---- текстовые сообщения: конфиг ----
            if "text" in msg and msg["text"] is not None:
                try:
                    data = json.loads(msg["text"])
                    if data.get("type") == "config":
                        src_sr = int(data.get("sample_rate", TARGET_SR))
                        packet_samples = int(data.get("packet_samples", 0)) or None
                        push_debug(
                            "info",
                            f"ws_audio sample_rate={src_sr}, packet_samples={packet_samples}"
                        )
                    if data.get("type") == "direction":
                        TRANSLATE_DIRECTION = data.get("value", "ru-en")

                        push_debug(
                            "info",
                            f"Translation direction: {TRANSLATE_DIRECTION}"
                        )
                except Exception as e:
                    push_debug("error", f"Audio config parse error: {e}")
                continue

            # ---- бинарные сообщения: PCM float32 mono ----
            if "bytes" not in msg or msg["bytes"] is None:
                continue

            raw = msg["bytes"]
            chunk = bytes_to_float32_pcm(raw)

            if chunk.size == 0:
                continue

            if src_sr != TARGET_SR:
                chunk = resample_audio(chunk, src_sr, TARGET_SR)

            received_samples += chunk.size

            # if received_samples % TARGET_SR < chunk.size:
            #     push_debug("info", f"rx_audio={received_samples / TARGET_SR:.2f}s")

            try:
                audio_q.put_nowait(chunk)
            except queue.Full:
                push_debug("error", "audio_q full; chunk dropped")

    except Exception as e:
        push_debug("error", f"Audio socket error: {e}")
    finally:
        push_debug("info", "Audio socket disconnected")


@app.post("/upload_reference")
async def upload_reference(request: Request):

    try:
        data = await request.body()

        if not data:
            push_debug("error", "Reference upload empty")
            return {"ok": False}

        with open(SPEAKER_REF, "wb") as f:
            f.write(data)

        push_debug("info", "Reference voice saved")

        return {"ok": True}

    except Exception as e:
        push_debug("error", f"Reference save error: {e}")
        return {"ok": False}

/tmp/ipykernel_60774/1698056691.py:725: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
/tmp/ipykernel_60774/1698056691.py:746: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("shutdown")


<<< Запуск через localhost.run

In [ ]:
import asyncio
import subprocess
import threading

# Функция для запуска SSH туннеля
def start_tunnel():
    # Команда для localhost.run
    cmd = [
        "ssh",
        "-o", "StrictHostKeyChecking=no",
        "-o", "ServerAliveInterval=60",  # Держим соединение живым
        "-R", "80:localhost:8000",
        "nokey@localhost.run"
    ]

    # Запускаем процесс и читаем вывод
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1
    )

    # Читаем вывод построчно
    for line in process.stdout:
        print(f"{line.strip()}")

# Запускаем FastAPI сервер
async def main():
    # Запускаем туннель в отдельном потоке
    tunnel_thread = threading.Thread(target=start_tunnel, daemon=True)
    tunnel_thread.start()

    # Даем время туннелю установиться
    await asyncio.sleep(2)

    # Запускаем сервер
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()

# Запускаем
await main()